# Mol property vs affinity correlation plots

Loads both datasets, computes 12 physicochemical properties per unique ligand, then plots correlations with pChEMBL value.

In [ ]:
import multiprocessing
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from scipy import stats
from tqdm import tqdm

RDLogger.DisableLog("rdApp.*")

repo_root = Path("..")
fig_dir = repo_root / "figures"
fig_dir.mkdir(exist_ok=True)

## Mol property computation

In [ ]:
FEATURE_NAMES = [
    "MolLogP",
    "ExactMolWt",
    "TPSA",
    "NumHDonors",
    "NumHAcceptors",
    "NumRotatableBonds",
    # "FormalCharge",
    "MolMR",
    "FractionCSP3",
    "RingCount",
    "NumAromaticRings",
    "HeavyAtomCount",
]

FEATURE_LABELS = {
    "MolLogP":           "Lipophilicity (MolLogP)",
    "ExactMolWt":        "Mol. weight (Da)",
    "TPSA":              "TPSA (\\AA$^2$)",
    "NumHDonors":        "H-bond donors",
    "NumHAcceptors":     "H-bond acceptors",
    "NumRotatableBonds": "Rotatable bonds",
    # "FormalCharge":      "Formal charge",
    "MolMR":             "Molar refractivity",
    "FractionCSP3":      "Fraction sp$^3$ C",
    "RingCount":         "Ring count",
    "NumAromaticRings":  "Aromatic ring count",
    "HeavyAtomCount":    "Heavy atom count",
}

FEATURE_LABELS_A = [v.replace("\\AA", "Å") for v in FEATURE_LABELS.values()]


def _compute_one(args: tuple) -> tuple | None:
    """Worker function: accepts (name, smi) tuple for multiprocessing compatibility."""
    name, smi = args
    RDLogger.DisableLog("rdApp.*")
    mol = Chem.MolFromSmiles(smi, sanitize=True)
    if mol is None:
        print(f"WARNING: could not parse SMILES, skipping. name={name}, smiles={smi}")
        return None
    mol = rdMolStandardize.LargestFragmentChooser().choose(mol)
    mol = Chem.RemoveAllHs(mol)
    props = np.array(
        [
            Descriptors.MolLogP(mol),
            Descriptors.ExactMolWt(mol),
            Descriptors.TPSA(mol),
            Descriptors.NumHDonors(mol),
            Descriptors.NumHAcceptors(mol),
            Descriptors.NumRotatableBonds(mol),
            # float(Chem.rdmolops.GetFormalCharge(mol)),
            Descriptors.MolMR(mol),
            Descriptors.FractionCSP3(mol),
            Descriptors.RingCount(mol),
            float(rdMolDescriptors.CalcNumAromaticRings(mol)),
            float(mol.GetNumHeavyAtoms()),
        ],
        dtype=np.float32,
    )
    return name, props


def compute_mol_properties(mol_names: list, smiles: list, n_jobs: int = 1) -> pd.DataFrame:
    """Compute mol properties for (name, smiles) pairs. Returns a DataFrame
    indexed by mol_name with FEATURE_NAMES as columns."""
    pairs = list(zip(mol_names, smiles))
    if n_jobs == 1:
        results = [_compute_one(p) for p in tqdm(pairs, desc="Computing mol props")]
    else:
        with multiprocessing.Pool(processes=n_jobs) as pool:
            results = list(tqdm(
                pool.imap(_compute_one, pairs, chunksize=256),
                total=len(pairs),
                desc=f"Computing mol props ({n_jobs} workers)",
            ))
    rows = [
        {"ligand_id": r[0], **dict(zip(FEATURE_NAMES, r[1]))}
        for r in results if r is not None
    ]
    return pd.DataFrame(rows).set_index("ligand_id")

## Load and prepare datasets

In [ ]:
# --- full_dataset (timesplit activities) ---
full_dataset = pd.read_parquet(repo_root / "../timesplit-affinity-benchmark/out/activities.parquet")
print(f"full_dataset: {full_dataset.shape}")

# Add target info data
target_df = pd.read_parquet(repo_root / "../timesplit-affinity-benchmark/out/targets.parquet")
full_dataset = full_dataset.join(
    target_df.set_index("target_chembl_id")[["uniprot_id", "gene_name"]],
    on="target_chembl_id"
)
# Add FEP+ 4 data split column
full_dataset = full_dataset.drop(columns=["split"]).join(
    pd.read_csv(repo_root / "data/out/FEP_data_split.csv").set_index("uniprot_id"),
    on="uniprot_id"
)

# Compute mol props once per unique ligand
unique_ligands = full_dataset.drop_duplicates(subset="ligand_chembl_id")[["ligand_chembl_id", "canonical_smiles"]].dropna()
print(f"Unique ligands in full_dataset: {len(unique_ligands)}")

full_dataset_props = compute_mol_properties(
    unique_ligands["ligand_chembl_id"].tolist(),
    unique_ligands["canonical_smiles"].tolist(),
    n_jobs=8,
)

# Join back onto full dataset
full_dataset = full_dataset.join(full_dataset_props, on="ligand_chembl_id")
print(f"full_dataset after mol prop join: {full_dataset.shape}")

In [ ]:
# --- FEPp_benchmark.csv ---
fepp = pd.read_csv(repo_root / "data/out/FEPp_benchmark.csv")
print(f"FEPp benchmark: {fepp.shape}")

# Small dataset — compute props directly for each row (no dedup needed)
fepp_props = compute_mol_properties(
    fepp["smiles"].tolist(),
    fepp["smiles"].tolist(),
)
fepp = fepp.join(fepp_props, on="smiles")
print(f"FEPp benchmark after mol prop join: {fepp.shape}")

## Tables and plots

In [ ]:
# Table for latex
latex_table = []
for feature_name in FEATURE_NAMES:
    df_filter_1 = full_dataset[
        (full_dataset["standard_type"] == "IC50")
        & (full_dataset["pchembl_relation"] == "=")
        & (full_dataset["data_split"] == "train")
        & full_dataset["pchembl_value_filled"].notna()
        & full_dataset[feature_name].notna()
    ].copy()
    y_min = df_filter_1[feature_name].quantile(0.025)
    y_max = df_filter_1[feature_name].quantile(0.975)
    df_filtered = df_filter_1.loc[
        df_filter_1[feature_name].apply(lambda y: y_min < y < y_max)
    ].copy()
    x = df_filtered[feature_name].values
    y = df_filtered["pchembl_value_filled"].values
    try:
        r, pval = stats.pearsonr(x, y)
    except ValueError as e:
        r, pval = np.nan, np.nan
    print(f"{feature_name:<20}\tPearson r = {r:.3f}, p = {pval:.2e}, n = {len(x):,}")
    latex_table.append(f"{FEATURE_LABELS[feature_name]} & {r:.3f} \\\\")
print("Latex table")
print("\n".join(latex_table))

In [ ]:
# Corrplot for latex
fepp_latex = fepp.copy()
remap_gene_names = {
    "CDK2": "CDK2",
    "TYK2": "TYK2",
    "MAPK8": "JNK1",
    "MAPK14": "p38$\\alpha$"
}
fepp_latex["gene_name"] = fepp_latex["gene_name"].map(remap_gene_names)
targets = fepp_latex["gene_name"].unique()
corr_data = {}
for gene in targets:
    df_t = fepp_latex[fepp_latex["gene_name"] == gene]
    corr_data[gene] = [
        df_t[[feat, "pchembl_value"]].dropna().corr().loc[feat, "pchembl_value"]
        for feat in FEATURE_NAMES
    ]
corr_df = pd.DataFrame(corr_data, index=FEATURE_LABELS_A)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list(
    "navy_white_red", ["#3a6ea5", "#ffffff", "#c0392b"]
)

# Version 1: annotated r values, FEATURE_NAMES labels, compact
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    corr_df.T,
    annot=True, fmt=".2f",
    cmap=cmap, vmin=-1, vmax=1,
    square=True,
    linewidths=0.4, linecolor="#eeeeee",
    annot_kws={"size": 9, "color": "#1a3a5c"},
    cbar_kws={"shrink": 1.0, "label": "Correlation ($r$)"},
    ax=ax,
)
ax.set_xlabel("", fontsize=11)
ax.set_ylabel("", fontsize=11)
ax.tick_params(axis="x", labelsize=9, rotation=90)
ax.tick_params(axis="y", labelsize=9, rotation=0)
ax.set_title("Molecular properties vs affinity (FEP+ 4 benchmark data)", fontsize=11)
for spine in ["top", "right", "left", "bottom"]:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig(fig_dir / "FEPp_mol_props_corrplot.png", dpi=300, bbox_inches="tight")
plt.show()
